**Case ID:** GL-0001

**Query:** Create a new customer in Stripe for all the buyers in this file.

**Context:** `Small-Online-Shop/buyers.json` is the current file.

**Case description:** The file maps buyer emails to names. None of those emails exist in Stripe at the start. At the end, every email must exist as a Stripe customer.

The model loop below runs the query against the tool specs, then the assertions grade the resulting state.

In [ ]:
BACKEND = "scripted"     # scripted | gemini
MODEL = "gemini-2.5-pro"
MAX_TURNS = 10
SKIP_BUYERS = 0          # scripted only: leave out N buyers to see the final assertion fail

# Set Up

## Simulated tools

In-memory `copilot` and `stripe` stand-ins, so the case needs no external services.

In [ ]:
import json
import os
import re
import time
from types import SimpleNamespace

BUYERS_FILE_PATH = "Small-Online-Shop/buyers.json"
buyers_data = {
    "olivia.martinez@emailprovider.com": "Olivia Martinez",
    "ethan.wilson@emailprovider.net": "Ethan Wilson",
    "ava.davis@emailprovider.org": "Ava Davis",
    "jackson.lee@emailprovider.com": "Jackson Lee",
    "sophia.rodriguez@emailprovider.net": "Sophia Rodriguez",
}
FILES = {BUYERS_FILE_PATH: json.dumps(buyers_data, indent=2)}
CUSTOMERS = []


def list_dir(path):
    prefix = path.rstrip("/") + "/"
    return [{"name": p[len(prefix):], "type": "file"} for p in FILES if p.startswith(prefix)]


def read_file(file_path, start_line, end_line):
    if file_path not in FILES:
        raise FileNotFoundError(file_path)
    lines = FILES[file_path].splitlines()
    return {"file_details": {
        "file_path": file_path,
        "content": "\n".join(lines[start_line - 1:end_line]),
        "start_line": start_line,
        "end_line": min(end_line, len(lines)),
        "total_lines": len(lines),
    }}


def create_customer(name, email=None):
    customer = {"id": f"cus_{len(CUSTOMERS) + 1}", "object": "customer", "name": name,
                "email": email, "created": int(time.time()), "livemode": False, "metadata": {}}
    CUSTOMERS.append(customer)
    return customer


def list_customers(limit=10, email=None):
    data = [c for c in CUSTOMERS if email is None or c["email"] == email]
    return {"object": "list", "data": data[:limit], "has_more": len(data) > limit}


copilot = SimpleNamespace(list_dir=list_dir, read_file=read_file)
stripe = SimpleNamespace(create_customer=create_customer, list_customers=list_customers)
TOOLS = {"read_file": read_file, "create_customer": create_customer, "list_customers": list_customers}

## Tool specs

Input definitions for the three tools the model can call.

In [ ]:
TOOL_SPECS = [
    {
        "name": "read_file",
        "description": """Read the contents of a file.

        This function reads a specified range of lines from a file. If the requested
        range is part of a larger file, an outline of the file structure may be
        provided. If the returned content is insufficient, this function can be
        called again to retrieve more content.""",
        "parameters": {
            "type": "object",
            "properties": {
                "file_path": {"type": "string", "description": "The path to the file to be read."},
                "start_line": {"type": "integer", "description": "The 1-based starting line number of the content to retrieve."},
                "end_line": {"type": "integer", "description": "The 1-based ending line number of the content to retrieve."},
            },
            "required": ["file_path", "start_line", "end_line"],
        },
    },
    {
        "name": "create_customer",
        "description": "This tool creates a customer in Stripe.",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "The name of the customer.", "minLength": 1, "maxLength": 200},
                "email": {"type": "string", "nullable": True, "default": None, "description": "The email of the customer.", "maxLength": 512},
            },
            "required": ["name"],
        },
    },
    {
        "name": "list_customers",
        "description": """This function fetches a list of Customers from Stripe. It processes an optional `limit`

        to control the number of customers retrieved and an optional `email` to filter
        customers by their email address in a case-sensitive manner.""",
        "parameters": {
            "type": "object",
            "properties": {
                "limit": {"type": "integer", "default": 10, "description": "A limit on the number of objects to be returned. Limit can range between 1 and 100. Defaults to 10."},
                "email": {"type": "string", "nullable": True, "default": None, "description": "A case-sensitive filter on the list based on the customer's email field. The value must be a string. Defaults to None.", "maxLength": 512},
            },
            "required": [],
        },
    },
]

## Assertion helper

List comparison with string normalization, covering the `all` and `any` cases.

In [ ]:
def normalize_string(string1):
    return (str(string1)
            .replace(".", "")
            .replace(",", "")
            .replace(";", "")
            .replace(":", "")
            .strip()
            .lower())


def compare_is_list_subset(search_value, input_list, list_comparison_function="all"):
    if not isinstance(input_list, list):
        raise TypeError("input_list must be a list")
    if len(input_list) == 0:
        raise ValueError("input_list must not be empty")
    if len(search_value) == 0:
        raise ValueError("search_value must not be empty")
    input_list = [normalize_string(item) for item in input_list]
    search_value = [normalize_string(item) for item in search_value]
    if list_comparison_function == "all":
        return all(item in input_list for item in search_value)
    return any(item in input_list for item in search_value)

# Initial Assertion

1. Assert that within the "Small-Online-Shop" directory, there is exactly one file named `buyers.json`.
2. Assert that the JSON file contains at least one valid email address as the key.
3. Assert that there are no customers in Stripe corresponding to any of the email addresses in the `buyers.json` file.

In [ ]:
dir_contents = copilot.list_dir(path="Small-Online-Shop")
buyers_json_files_count = sum(1 for item in dir_contents if item["name"] == "buyers.json" and item["type"] == "file")
assert buyers_json_files_count == 1, f"Expected 1 'buyers.json' file, found {buyers_json_files_count}"

content_str = copilot.read_file(file_path=BUYERS_FILE_PATH, start_line=1, end_line=1000)["file_details"]["content"]
all_keys_from_json = list(json.loads(content_str).keys())
email_regex = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"
assert any(re.match(email_regex, key) for key in all_keys_from_json), \
    f"'{BUYERS_FILE_PATH}' does not contain at least one valid email address as a key."

found = [key for key in all_keys_from_json if stripe.list_customers(email=key)["data"]]
assert not found and len(all_keys_from_json) > 0, f"Found Stripe customer(s) for: {found}"
print("Initial assertion passed")

# Action

The model sees the query and the tool specs, and can only change state through tool calls. Each backend takes the previous tool results and returns the next tool calls plus any text.

- `scripted` replays the correct calls, so the loop and assertions run with no API key.
- `gemini` calls the real model. It reads the key from `GEMINI_API_KEY` or `GOOGLE_API_KEY`.

In [ ]:
QUERY = "Create a new customer in Stripe for all the buyers in this file."
PROMPT = f"""{QUERY}

<additional_data>
<current_file>Path: {BUYERS_FILE_PATH}</current_file>
</additional_data>"""


def scripted_backend():
    buyers = list(buyers_data.items())[SKIP_BUYERS:]
    turns = iter([
        [("read_file", {"file_path": BUYERS_FILE_PATH, "start_line": 1, "end_line": 1000})],
        [("create_customer", {"name": name, "email": email}) for email, name in buyers],
    ])

    def step(results):
        calls = next(turns, [])
        return calls, "" if calls else "Created the customers."
    return step


def gemini_backend():
    from google import genai
    from google.genai import types

    api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        raise RuntimeError("Set GEMINI_API_KEY or GOOGLE_API_KEY to use the gemini backend")
    client = genai.Client(api_key=api_key)
    config = types.GenerateContentConfig(
        tools=[types.Tool(function_declarations=[types.FunctionDeclaration(**spec) for spec in TOOL_SPECS])],
    )
    history = [types.Content(role="user", parts=[types.Part(text=PROMPT)])]

    def step(results):
        if results:
            history.append(types.Content(role="user", parts=[
                types.Part.from_function_response(name=name, response={"result": result})
                for name, result in results
            ]))
        response = client.models.generate_content(model=MODEL, contents=history, config=config)
        content = response.candidates[0].content
        history.append(content)
        parts = content.parts or []
        calls = [(p.function_call.name, dict(p.function_call.args or {})) for p in parts if p.function_call]
        return calls, "".join(p.text for p in parts if p.text)
    return step


def run_agent(step):
    results, trace = None, []
    for _ in range(MAX_TURNS):
        calls, text = step(results)
        if not calls:
            return text, trace
        results = []
        for name, args in calls:
            try:
                result = TOOLS[name](**args)
            except Exception as e:
                result = {"error": f"{type(e).__name__}: {e}"}
            trace.append((name, args))
            results.append((name, result))
    raise RuntimeError(f"Model did not finish within {MAX_TURNS} turns")


step = scripted_backend() if BACKEND == "scripted" else gemini_backend()
final_text, trace = run_agent(step)
for name, args in trace:
    print(name, args)
print("Model:", final_text)

# Final Assertion

1. Assert that there exists a `customer` object in Stripe for each of the email addresses in the `Small-Online-Shop/buyers.json` file.

In [ ]:
file_response = copilot.read_file(file_path=BUYERS_FILE_PATH, start_line=1, end_line=1000)
content = file_response.get("file_details", {}).get("content", "")
buyers = json.loads(content) if content else {}

stripe_emails = [customer["email"] for customer in stripe.list_customers().get("data")]
json_emails = list(buyers.keys())

assert compare_is_list_subset(json_emails, stripe_emails, "all"), \
    f"Expected all emails from json to be in stripe, expected: {json_emails}, \n actual: {stripe_emails}"
print("Final assertion passed")